Q1:What is the total net revenue for customers in region West after subtracting returned units at the original discounted unit price, rounded to 2 decimals?

In [2]:
import pandas as pd

In [3]:
from pathlib import Path
base = Path("../../tasks/task_01/data/")

In [4]:
customer_df = pd.read_csv(base/ "customers.csv")
product_df = pd.read_csv(base/ "products.csv")
order_df = pd.read_csv(base/ "orders.csv")
returns_df = pd.read_csv(base/ "returns.csv")

In [5]:
print(customer_df.head())
print(product_df.head())
print(order_df.head())
print(returns_df.head())

  customer_id   region
0         C01     East
1         C02  Central
2         C03    South
3         C04     West
4         C05    North
  product_id category  unit_price
0        P01     Home        12.5
1        P02   Office        18.0
2        P03  Outdoor        23.0
3        P04  Kitchen        31.0
4        P05     Home        14.0
  order_id  order_date customer_id product_id  quantity  discount_pct  channel
0    O0001  2024-01-01         C01        P05         2            10    store
1    O0002  2024-01-01         C02        P08         3            15  partner
2    O0003  2024-01-01         C03        P03         4             0      web
3    O0004  2024-01-01         C04        P06         1             5    store
4    O0005  2024-01-01         C05        P01         2            10  partner
  return_id order_id return_date  returned_qty        reason
0      R001    O0009  2024-01-03             1  changed_mind
1      R002    O0018  2024-01-04             1       damaged
2

In [6]:
orders = order_df.merge(customer_df, on="customer_id").merge(product_df, on="product_id")
returned = returns_df.groupby("order_id", as_index=False)["returned_qty"].sum()
orders = orders.merge(returned, on="order_id", how="left").fillna({"returned_qty": 0})
orders["net_revenue"] = (orders["quantity"] - orders["returned_qty"]) * orders["unit_price"] * (1 - orders["discount_pct"] / 100)
west_net_revenue = orders.loc[orders['region'] == 'West', 'net_revenue'].sum()
print(f"{west_net_revenue:.2f}")

1521.95


Q2: Which customer_id has the highest net revenue from Office products sold through the store channel on odd-numbered January dates?

In [7]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

q2_frame = orders[
    (orders["category"] == "Office")
    & (orders["channel"] == "store")
    & (orders["order_date"].dt.day % 2 == 1)
]
highest_rev = str(q2_frame.groupby("customer_id")["net_revenue"].sum().sort_values(ascending=False).index[0])
print(highest_rev)

C02


Q3: On which date did total net revenue have the largest increase versus the previous date?

In [8]:
daily = orders.groupby("order_date", as_index=False)["net_revenue"].sum().sort_values("order_date")
daily["change"] = daily["net_revenue"].diff()
max_change_date = daily.loc[daily["change"].idxmax(), "order_date"].date().isoformat()
print(max_change_date)

2024-01-27


Q4: Which product_id has the highest return rate by units returned divided by units sold?

In [9]:
return_rate = (
    orders.groupby("product_id")["returned_qty"].sum()
    / orders.groupby("product_id")["quantity"].sum()
).sort_values(ascending=False)
highest_return_rate = str(return_rate.index[0])
print(highest_return_rate)


P02


Q5: For customer C07, what is the median number of days between consecutive orders?

In [10]:
c07_dates = orders.loc[orders["customer_id"] == "C07", "order_date"].drop_duplicates().sort_values()
median_days = str(int(c07_dates.diff().dropna().dt.days.median()))
print(median_days)

1
